# Diabetes classification v2

The original experiment trained on 80 rows and tuned for recall. In its saved results, tuning raised specificity but lowered sensitivity at the default threshold. This version tests three changes: using more training data, searching for models with better precision–recall ranking, and choosing a cutoff using development data.

The final test rows are held out from imputation, model selection, and cutoff selection. Compare all variants on these same rows. The original notebook's reported scores used a different split.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    average_precision_score,
    confusion_matrix,
    precision_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    TunedThresholdClassifierCV,
    train_test_split,
)
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

## 1. Load and split the data

Set aside 20% of the rows as a stratified test set. The 80-row baseline draws its training sample from the development set, so all models use the same test rows. Zeros in five measurement columns represent missing values. Median imputation is fit within each training fold.

In [ ]:
data_path = Path("data/diabetes.csv")
if not data_path.is_file():
    raise FileNotFoundError(f"Place the project CSV at {data_path} before running the notebook.")

features = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

data = pd.read_csv(data_path)
X = data[features].copy()
X[zero_as_missing] = X[zero_as_missing].replace(0, np.nan)
y = data["Outcome"]
if y.isna().any() or set(y.unique()) != {0, 1}:
    raise ValueError("Outcome must contain both binary labels (0 and 1) and no missing values.")

seed = 42
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=seed
)
X_small, _, y_small, _ = train_test_split(
    X_dev, y_dev, train_size=80, stratify=y_dev, random_state=seed
)
print(f"Development rows: {len(X_dev)}, test rows: {len(X_test)}")

## 2. Establish two baselines

Fit the same default XGBoost pipeline on 80 development rows and on the full development set. This estimates how much the extra training data helps before any tuning.

In [ ]:
def default_xgb():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        XGBClassifier(random_state=seed, n_jobs=1),
    )

small_baseline = default_xgb().fit(X_small, y_small)
full_baseline = default_xgb().fit(X_dev, y_dev)

## 3. Select the model and classification threshold

The inner cross-validation search compares regularized logistic regression and shallow XGBoost models using average precision (AP). AP summarizes precision across levels of recall. It helps reveal whether a model ranks positives well without first choosing a classification threshold.

The outer cross-validation loop fits that search on each training fold and scores thresholds on the corresponding validation fold. It selects the threshold that maximizes mean balanced accuracy, the average of sensitivity and specificity. This gives both classes equal weight when there is no specified cost for a false positive or false negative. The model search then refits on all development rows.

In [ ]:
pipeline = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", "passthrough"),
        ("model", LogisticRegression()),
    ]
)
candidates = [
    {
        "scale": [StandardScaler()],
        "model": [LogisticRegression(max_iter=1000)],
        "model__C": [0.1, 1.0, 10.0],
        "model__class_weight": [None, "balanced"],
    },
    {
        "scale": ["passthrough"],
        "model": [
            XGBClassifier(
                random_state=seed,
                n_jobs=1,
                n_estimators=150,
                subsample=0.8,
                colsample_bytree=0.8,
            )
        ],
        "model__max_depth": [2, 3, 4],
        "model__learning_rate": [0.05, 0.1],
        "model__scale_pos_weight": [1.0, 2.0],
    },
]
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed + 1)
search = GridSearchCV(
    pipeline,
    candidates,
    scoring="average_precision",
    cv=inner_cv,
    error_score="raise",
)
cutoff_search = TunedThresholdClassifierCV(
    search,
    scoring="balanced_accuracy",
    cv=outer_cv,
    thresholds=100,
)
cutoff_search.fit(X_dev, y_dev)

fitted_search = cutoff_search.estimator_
chosen_threshold = cutoff_search.best_threshold_
cv_results = pd.DataFrame(fitted_search.cv_results_)
cv_results["Model"] = cv_results["param_model"].map(lambda model: type(model).__name__)
best_ap_by_model = cv_results.groupby("Model")["mean_test_score"].max()

print("Selected model:", type(fitted_search.best_estimator_.named_steps["model"]).__name__)
selected_parameters = {
    key.removeprefix("model__"): value
    for key, value in fitted_search.best_params_.items()
    if key.startswith("model__")
}
print("Selected hyperparameters:", selected_parameters)
print(f"Selected threshold: {chosen_threshold:.4f}")
print(f"Development CV balanced accuracy at selected threshold: {cutoff_search.best_score_:.4f}")
best_ap_by_model.rename("Best development CV average precision").round(4)

## 4. Evaluate on the held-out test set

Compare the selected model at 0.50 and at the tuned cutoff. This isolates the effect of changing the classification decision. AP and ROC-AUC depend on the scores and will be identical for these two rows. Use the confusion counts to see how the cutoff affects missed positives and false positives.

In [ ]:
def evaluate(y_true, scores, cutoff):
    predicted = scores >= cutoff
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return {
        "Threshold": cutoff,
        "Average precision": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "Balanced accuracy": (sensitivity + specificity) / 2,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision_score(y_true, predicted, zero_division=0),
        "False positives": fp,
        "False negatives": fn,
    }

test_scores = {
    "XGBoost baseline (80 rows)": small_baseline.predict_proba(X_test)[:, 1],
    "XGBoost baseline (full development set)": full_baseline.predict_proba(X_test)[:, 1],
    "Selected model": fitted_search.predict_proba(X_test)[:, 1],
}
results = {name: evaluate(y_test, scores, 0.5) for name, scores in test_scores.items()}
results["Selected model (tuned cutoff)"] = evaluate(
    y_test, test_scores["Selected model"], chosen_threshold
)
pd.DataFrame.from_dict(results, orient="index").round(4)

In [ ]:
fig, (pr_ax, roc_ax) = plt.subplots(1, 2, figsize=(13, 5), layout="constrained")

for name, scores in test_scores.items():
    PrecisionRecallDisplay.from_predictions(y_test, scores, name=name, ax=pr_ax)
    RocCurveDisplay.from_predictions(y_test, scores, name=name, ax=roc_ax)

pr_ax.axhline(y_test.mean(), color="gray", linestyle="--", label="Positive prevalence")
pr_ax.legend()
roc_ax.plot([0, 1], [0, 1], color="gray", linestyle="--")
plt.show()

## Interpret the comparison

- Compare the two baseline rows to estimate the benefit of using more training data.
- Compare the full XGBoost baseline and the selected model on test AP and ROC-AUC to see whether model selection improved ranking.
- Compare the selected model at both thresholds to see whether a gain in sensitivity comes with more false positives or a drop in precision.
- A single split on a small dataset gives uncertain estimates. If you use these test results to make further modeling decisions, use a new holdout or external dataset for the next final comparison. Class weighting can affect probability calibration. The scores are for this classification experiment.

## Reflection and next experiment

This generation separates the effect of more training rows from the effect of changing the cutoff. On the shared holdout, lowering the logistic regression cutoff caught more positives and produced more false positives. The third generation asks how those conclusions change when overall accuracy is the primary endpoint and the search has a fixed fit budget.